## 1. Setup & Parse Arguments

In [1]:
# Auto-reload modules
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from argparse import Namespace
import sys
from datetime import datetime
import gc
import torch
from train import train

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 5080
GPU Memory: 17.09 GB


In [ ]:
from config import (
    BATCH_SIZE, NUM_EPOCHS, LEARNING_RATE,
    MODE, FUSION_METHOD, FEATURE_MODE,
    CHECKPOINT_DIR, BEST_MODEL_NAME,
)

# ============================================================================
# CẤU HÌNH GRID SEARCH CHO DỰ ÁN DUAL-STREAM (FBANK + HANDCRAFTED)
# ============================================================================
ROOT_DIR = r"E:\speech_data\features\train" # Thư mục gốc chứa các thư mục duration
durations = ['train_raw', 'train_vi_full']  # 'train_raw', 'train_vi_3s', 'train_vi_5s', 'train_vi_7s', 'train_vi_full'

# Các feature mode (Dựa vào config.py của bạn)
feature_modes = ["mfbe_pitch", "mfbe_only", "pitch_only"]
fusion_methods = ["concat", "cross_attention", "gating"]

def get_base_args():
    return Namespace(
        test_dir="D:/test", # Thay đổi nếu cần
        output_dir="./outputs",
        batch_size=64, # Dự án này dùng Conv1D nhỏ nên 64-128 là đẹp
        learning_rate=0.0001,
        lr_scheduler="plateau",
        num_epochs=50,
        optimizer="adam",
        weight_decay=0.001,
        aam_margin=0.3,
        aam_scale=30,
        mixed_precision=True,
        early_stop_patience=5,
        seed=42,
        device="cuda",
        
        # Biến gán động
        mode=3,
        exp_name="",
        base_dir="",
        feature_mode="",
        fusion_method="",
        duration=""
    )

def check_skip(exp_name):
    if os.path.exists(os.path.join("./outputs/experiments", exp_name, "results.json")):
        print(f"⏭ Bỏ qua: {exp_name} - Đã xong!")
        return True
    return False

In [ ]:
import os
import random
from collections import defaultdict

def create_fixed_trials(val_dir, output_file, num_pairs=20000):
    # Gom tất cả file theo Speaker ID
    speaker_to_files = defaultdict(list)
    for root, _, files in os.walk(val_dir):
        for f in files:
            if f.endswith('.pt'):
                spk_id = f.split('_')[0] # Giả sử tên file có ID người nói ở đầu
                speaker_to_files[spk_id].append(os.path.join(root, f))

    all_speakers = list(speaker_to_files.keys())
    trials = []

    # 1. Tạo Positive Pairs (Target)
    for _ in range(num_pairs // 2):
        spk = random.choice([s for s in all_speakers if len(speaker_to_files[s]) >= 2])
        f1, f2 = random.sample(speaker_to_files[spk], 2)
        trials.append(f"1 {f1} {f2}")

    # 2. Tạo Negative Pairs (Non-target)
    for _ in range(num_pairs // 2):
        spk1, spk2 = random.sample(all_speakers, 2)
        f1 = random.choice(speaker_to_files[spk1])
        f2 = random.choice(speaker_to_files[spk2])
        trials.append(f"0 {f1} {f2}")

    random.shuffle(trials)
    with open(output_file, 'w') as f:
        f.write('\n'.join(trials))
    print(f"✓ Đã tạo file trials cố định tại: {output_file}")

print("Kiểm tra và tạo file val_trials.txt cố định cho các tập dữ liệu...")
for dur in durations:
    val_dir = os.path.join(ROOT_DIR, dur)
    trial_file = os.path.join(val_dir, "val_trials.txt")
    if not os.path.exists(trial_file):
        print(f" -> Đang tạo trials cho {dur}...")
        create_fixed_trials(val_dir, trial_file, num_pairs=20000)
print("✓ Đã chuẩn bị xong toàn bộ Trials!")

## 2. Training

In [ ]:
# 1. MODE 1: CHỈ DÙNG FBANK (5 Experiments)
for dur in durations:
    exp_name = f"Mode1_FBank_{dur}"
    if check_skip(exp_name): continue
    
    print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
    args = get_base_args()
    args.mode = 1
    args.exp_name = exp_name
    args.duration = dur
    args.base_dir = os.path.join(ROOT_DIR, dur)
    args.feature_mode = "N/A"
    args.fusion_method = "N/A"
    
    train(args)
    gc.collect(); torch.cuda.empty_cache()

# 2. MODE 2: CHỈ DÙNG HANDCRAFTED (15 Experiments)
for dur in durations:
    for feat in feature_modes:
        exp_name = f"Mode2_HC_{dur}_{feat}"
        if check_skip(exp_name): continue
        
        print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
        args = get_base_args()
        args.mode = 2
        args.exp_name = exp_name
        args.duration = dur
        args.base_dir = os.path.join(ROOT_DIR, dur)
        args.feature_mode = feat
        args.fusion_method = "N/A"
        
        train(args)
        gc.collect(); torch.cuda.empty_cache()

# 3. MODE 3: HYBRID FUSION (45 Experiments)
for dur in durations:
    for feat in feature_modes:
        for fusion in fusion_methods:
            exp_name = f"Mode3_{fusion}_{dur}_{feat}"
            if check_skip(exp_name): continue
            
            print(f"\n{'='*80}\nĐANG CHẠY: {exp_name}\n{'='*80}")
            args = get_base_args()
            args.mode = 3
            args.exp_name = exp_name
            args.duration = dur
            args.base_dir = os.path.join(ROOT_DIR, dur)
            args.feature_mode = feat
            args.fusion_method = fusion
            
            train(args)
            gc.collect(); torch.cuda.empty_cache()

print("\n🎉 TOÀN BỘ GRID SEARCH ĐÃ HOÀN TẤT THÀNH CÔNG!")

⏭ Bỏ qua: Mode1_FBank_train_raw - Đã xong!
⏭ Bỏ qua: Mode1_FBank_train_vi_full - Đã xong!

ĐANG CHẠY: Mode2_HC_train_raw_mfbe_pitch
🔍 Đang quét dữ liệu Train/Val tại: E:\speech_data\features\train\train_raw...
  -> Tối ưu RAM: Nạp thẳng data vào RAM (Float16) từ mfbe_pitch_shards...

Khởi tạo mô hình thành công (Hybrid ECAPA-TDNN)
  Mode: 2 (1=Main, 2=Handcrafted, 3=Fusion)
  Handcrafted Feature: MFBE_PITCH
  Num speakers: 2023
  Trainable parameters: 1,051,136

Starting training (Open-set Validation)...



  -> Đang tính toán EER (Open-set) cho Epoch 1...

[Evaluation] Extracting embeddings...


[Evaluation] Generating 20000 balanced pairs for scoring...
Epoch   1 | Train Loss: 8.9403, Acc: 0.1877 | Val EER: 17.39% | MinDCF: 0.0361 | LR: 0.000100


  -> Đang tính toán EER (Open-set) cho Epoch 2...

[Evaluation] Extracting embeddings...


[Evaluation] Generating 20000 balanced pairs for scoring...
Epoch   2 | Train Loss: 7.5275, Acc: 0.2599 | Val EER: 16.25% | MinDCF: 0.0325 | LR: 0.000100


...

  -> Đang tính toán EER (Open-set) cho Epoch 9...

[Evaluation] Extracting embeddings...


[Evaluation] Generating 20000 balanced pairs for scoring...
Epoch   9 | Train Loss: 8.1993, Acc: 0.0452 | Val EER: 34.45% | MinDCF: 0.0495 | LR: 0.000050


  -> Đang tính toán EER (Open-set) cho Epoch 10...

[Evaluation] Extracting embeddings...


d:\my_project\SLP301-data\SpeakerVeri_ECAPA\train\train.py:47: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location=DEVICE)


[Evaluation] Generating 20000 balanced pairs for scoring...
Epoch  10 | Train Loss: 7.7360, Acc: 0.0361 | Val EER: 35.38% | MinDCF: 0.0496 | LR: 0.000050

✓ Early stopping triggered do EER không giảm nữa!
⏭ Bỏ qua: Mode3_concat_train_raw_mfbe_pitch - Đã xong!

ĐANG CHẠY: Mode3_cross_attention_train_raw_mfbe_pitch
🔍 Đang quét dữ liệu Train/Val tại: E:\speech_data\features\train\train_raw...
  -> Tối ưu RAM: Nạp thẳng data vào RAM (Float16) từ fbank_shards...
  -> Tối ưu RAM: Nạp thẳng data vào RAM (Float16) từ mfbe_pitch_shards...


## 3. Training Curves

In [ ]:
# Load TensorBoard extension
%load_ext tensorboard

# Chạy giao diện TensorBoard trỏ vào thư mục chứa tất cả các experiments
%tensorboard --logdir ./outputs/experiments

## 4. Test Results

In [1]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

try:
    from model import get_model
    from metrics import compute_eer, compute_mindcf
except ImportError:
    from train.model import get_model
    from train.metrics import compute_eer, compute_mindcf

# ============================================================================
# 4. INFERENCE ALL MODELS ON TEST_O CSV PAIRS
# ============================================================================

EXPERIMENTS_DIR = "./outputs/experiments"
CSV_PAIRS_PATH = r"D:/SpeakerVeri_ECAPA/test_o/test_list_gt.csv"

# File FBank dùng chung cho mode 1 & mode 3 (để None nếu không đánh giá mode cần FBank)
FBANK_PT_PATH = r"D:/SpeakerVeri_ECAPA/test_o/test_O_fbank.pt"

# Mỗi feature_mode có thể dùng 1 file handcrafted khác nhau.
# Nếu bạn chỉ có 1 file cho tất cả, đặt cùng 1 đường dẫn cho 3 key.
HANDCRAFTED_PT_BY_MODE = {
    "mfbe_pitch": r"D:/SpeakerVeri_ECAPA/test_o/all_features_MFBE_Pitch.pt",
    "mfbe_only":  r"D:/SpeakerVeri_ECAPA/test_o/all_features_MFBE.pt",
    "pitch_only": r"D:/SpeakerVeri_ECAPA/test_o/all_features_Pitch.pt",
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_PAIRS_LIMIT = None   # None = chạy toàn bộ cặp trong CSV; hoặc set ví dụ 50000
P_TARGET = 0.05


def _load_pairs_csv(csv_path):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Không tìm thấy CSV: {csv_path}")

    raw = pd.read_csv(csv_path, header=None)
    if raw.shape[1] == 1:
        parts = raw[0].astype(str).str.strip().str.split(r"\s+", n=2, expand=True)
        if parts.shape[1] < 3:
            raise ValueError("CSV 1 cột nhưng không tách được đủ 3 trường: label path1 path2")
        df = pd.DataFrame({"label": parts[0], "path1": parts[1], "path2": parts[2]})
    else:
        df = pd.DataFrame({"label": raw.iloc[:, 0], "path1": raw.iloc[:, 1], "path2": raw.iloc[:, 2]})

    df["label"] = df["label"].astype(int)
    df["path1"] = df["path1"].astype(str).str.strip()
    df["path2"] = df["path2"].astype(str).str.strip()
    return df


def _normalize_feat_tensor(feat):
    if not torch.is_tensor(feat):
        feat = torch.tensor(feat)

    feat = feat.float()
    if feat.dim() == 1:
        feat = feat.unsqueeze(0)   # (C, T)
    elif feat.dim() == 3 and feat.shape[0] == 1:
        feat = feat.squeeze(0)      # (C, T)
    elif feat.dim() != 2:
        raise ValueError(f"Feature tensor shape không hợp lệ: {tuple(feat.shape)}")
    return feat


def _candidate_keys(path_text):
    p = str(path_text).replace('\\', '/').strip()
    base = os.path.basename(p)
    stem = os.path.splitext(base)[0]
    rel = p.lstrip('./')
    candidates = [p, rel, base, stem]
    # unique giữ thứ tự
    seen = set()
    out = []
    for k in candidates:
        if k not in seen:
            out.append(k)
            seen.add(k)
    return out


def _get_feature_from_dict(feat_dict, path_text):
    for key in _candidate_keys(path_text):
        if key in feat_dict:
            return feat_dict[key]
    return None


def _evaluate_one_model(
    model,
    mode,
    pairs_df,
    device,
    fbank_dict=None,
    hand_dict=None,
    p_target=0.05,
    num_pairs_limit=None,
):
    model.eval()
    rows = pairs_df if num_pairs_limit is None else pairs_df.iloc[:num_pairs_limit]

    cache = {}
    missing_pairs = 0
    scores = []
    labels = []

    def get_embedding(utt_path):
        if utt_path in cache:
            return cache[utt_path]

        inputs = {}
        if mode in [1, 3]:
            if fbank_dict is None:
                raise ValueError("Model mode cần FBank nhưng chưa cung cấp FBANK_PT_PATH")
            f_feat = _get_feature_from_dict(fbank_dict, utt_path)
            if f_feat is None:
                cache[utt_path] = None
                return None
            f_feat = _normalize_feat_tensor(f_feat).unsqueeze(0).to(device)
            inputs["fbank"] = f_feat

        if mode in [2, 3]:
            if hand_dict is None:
                raise ValueError("Model mode cần handcrafted nhưng chưa cung cấp file .pt phù hợp")
            h_feat = _get_feature_from_dict(hand_dict, utt_path)
            if h_feat is None:
                cache[utt_path] = None
                return None
            h_feat = _normalize_feat_tensor(h_feat).unsqueeze(0).to(device)
            inputs["handcrafted"] = h_feat

        with torch.no_grad():
            _, emb = model(**inputs)
            emb = F.normalize(emb, p=2, dim=1).squeeze(0).cpu()

        cache[utt_path] = emb
        return emb

    for row in tqdm(rows.itertuples(index=False), total=len(rows), leave=False):
        emb1 = get_embedding(row.path1)
        emb2 = get_embedding(row.path2)
        if emb1 is None or emb2 is None:
            missing_pairs += 1
            continue

        score = float(torch.dot(emb1, emb2).item())
        scores.append(score)
        labels.append(int(row.label))

    if len(scores) == 0:
        raise ValueError("Không có cặp hợp lệ nào để tính metric")

    eer, _ = compute_eer(labels, scores)
    mindcf, _ = compute_mindcf(labels, scores, p_target=p_target)

    return {
        "num_pairs_used": len(scores),
        "num_pairs_missing": missing_pairs,
        "eer_percent": float(eer * 100),
        "mindcf": float(mindcf),
    }


def run_all_experiments_inference():
    exp_root = Path(EXPERIMENTS_DIR)
    if not exp_root.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục experiments: {exp_root}")

    pairs_df = _load_pairs_csv(CSV_PAIRS_PATH)
    print(f"✅ Loaded CSV pairs: {len(pairs_df)} rows from {CSV_PAIRS_PATH}")

    fbank_dict = None
    if FBANK_PT_PATH and os.path.exists(FBANK_PT_PATH):
        fbank_dict = torch.load(FBANK_PT_PATH, map_location="cpu")
        if not isinstance(fbank_dict, dict):
            raise ValueError("FBANK_PT_PATH phải là dict key->tensor")
        print(f"✅ Loaded FBank dict: {len(fbank_dict)} entries")
    else:
        print("ℹ Không load FBank dict (sẽ skip model cần FBank nếu thiếu)")

    handcrafted_cache = {}
    results = []

    exp_dirs = sorted([d for d in exp_root.iterdir() if d.is_dir()])
    print(f"\n🚀 Bắt đầu inference cho {len(exp_dirs)} experiments...")

    for exp_dir in tqdm(exp_dirs, desc="Experiments"):
        config_path = exp_dir / "config.json"
        checkpoint_path = exp_dir / "best_model.pth"
        if not checkpoint_path.exists():
            checkpoint_path = exp_dir / "best_eer_model.pth"

        if (not config_path.exists()) or (not checkpoint_path.exists()):
            continue

        with open(config_path, "r", encoding="utf-8") as f:
            cfg = json.load(f)

        mode = int(cfg.get("mode", 3))
        feature_mode = str(cfg.get("feature_mode", "mfbe_pitch"))
        fusion_method = str(cfg.get("fusion_method", "concat"))

        hand_path = HANDCRAFTED_PT_BY_MODE.get(feature_mode)
        hand_dict = None
        if mode in [2, 3]:
            if not hand_path or (not os.path.exists(hand_path)):
                results.append({
                    "experiment": exp_dir.name,
                    "mode": mode,
                    "feature_mode": feature_mode,
                    "fusion_method": fusion_method,
                    "status": "skip_missing_handcrafted_file",
                })
                continue
            if hand_path not in handcrafted_cache:
                handcrafted_cache[hand_path] = torch.load(hand_path, map_location="cpu")
                if not isinstance(handcrafted_cache[hand_path], dict):
                    raise ValueError(f"File handcrafted không đúng định dạng dict: {hand_path}")
            hand_dict = handcrafted_cache[hand_path]

        if mode in [1, 3] and fbank_dict is None:
            results.append({
                "experiment": exp_dir.name,
                "mode": mode,
                "feature_mode": feature_mode,
                "fusion_method": fusion_method,
                "status": "skip_missing_fbank_file",
            })
            continue

        try:
            model = get_model(
                num_speakers=1,
                device=str(DEVICE),
                mode=mode,
                fusion_method=fusion_method,
                feature_mode=feature_mode,
            )
            ckpt = torch.load(checkpoint_path, map_location=DEVICE)
            state_dict = ckpt["model_state_dict"] if isinstance(ckpt, dict) and "model_state_dict" in ckpt else ckpt
            model.load_state_dict(state_dict, strict=True)

            metrics = _evaluate_one_model(
                model=model,
                mode=mode,
                pairs_df=pairs_df,
                device=DEVICE,
                fbank_dict=fbank_dict,
                hand_dict=hand_dict,
                p_target=P_TARGET,
                num_pairs_limit=NUM_PAIRS_LIMIT,
            )

            results.append({
                "experiment": exp_dir.name,
                "mode": mode,
                "feature_mode": feature_mode,
                "fusion_method": fusion_method,
                "checkpoint": checkpoint_path.name,
                "status": "ok",
                "eer_percent": round(metrics["eer_percent"], 4),
                "mindcf": round(metrics["mindcf"], 6),
                "pairs_used": metrics["num_pairs_used"],
                "pairs_missing": metrics["num_pairs_missing"],
            })

        except Exception as ex:
            results.append({
                "experiment": exp_dir.name,
                "mode": mode,
                "feature_mode": feature_mode,
                "fusion_method": fusion_method,
                "status": f"error: {type(ex).__name__}",
                "error_message": str(ex),
            })

    if not results:
        print("⚠ Không có experiment nào được đánh giá.")
        return None

    results_df = pd.DataFrame(results)
    ok_df = results_df[results_df["status"] == "ok"].copy()
    if len(ok_df) > 0:
        ok_df = ok_df.sort_values(["eer_percent", "mindcf"], ascending=[True, True])
        print("\n✅ TOP RESULTS (status=ok)")
        display(ok_df.head(20))
    else:
        print("⚠ Không có experiment nào chạy thành công.")

    save_path = exp_root / "inference_test_o_summary.csv"
    results_df.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"\n💾 Đã lưu bảng tổng hợp: {save_path}")

    return results_df


results_df = run_all_experiments_inference()

✅ Loaded CSV pairs: 240 rows from D:/SpeakerVeri_ECAPA/test_o/test_list_gt.csv
ℹ Không load FBank dict (sẽ skip model cần FBank nếu thiếu)

🚀 Bắt đầu inference cho 11 experiments...


Experiments:   0%|          | 0/11 [00:00<?, ?it/s]


Khởi tạo mô hình thành công (Hybrid ECAPA-TDNN)
  Mode: 2 (1=Main, 2=Handcrafted, 3=Fusion)
  Handcrafted Feature: PITCH_ONLY
  Num speakers: 1
  Trainable parameters: 1,020,416



Experiments: 100%|██████████| 11/11 [00:00<00:00, 31.64it/s]


✅ TOP RESULTS (status=ok)


,experiment,mode,feature_mode,fusion_method,checkpoint,status,eer_percent,mindcf,pairs_used,pairs_missing
0,smoke_mode2_pitch,2,pitch_only,concat,best_model.pth,ok,65.8333,0.875,240,0



💾 Đã lưu bảng tổng hợp: outputs\experiments\inference_test_o_summary.csv


## 5. Gating Analysis (Only for Mode 3 + Gating)

In [ ]:
from train import analyze_gating_behavior
import matplotlib.image as mpimg

# Kiểm tra điều kiện để chạy phân tích
if args.mode == 3 and args.fusion_method == "gating":
    print("Đang thực hiện phân tích cơ chế Gating trên tập Test...")
    
    # Gọi hàm phân tích từ train.py
    # Lưu ý: Hàm này trả về 2 giá trị: gates (trọng số PTM) và labels
    gates, labels = analyze_gating_behavior(model, test_loader, device, exp_dir)
    
    # Hiển thị biểu đồ phân phối trọng số đã được lưu
    gate_plot_path = os.path.join(exp_dir, "gating_analysis", "gate_distribution.png")
    if os.path.exists(gate_plot_path):
        plt.figure(figsize=(10, 6))
        img = mpimg.imread(gate_plot_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title("Phân phối trọng số Gating (Trục X > 0.5 ưu tiên PTM, < 0.5 ưu tiên Handcrafted)")
        plt.show()
        
    # In thông tin thống kê tóm tắt
    ptm_wins = np.sum(gates > 0.5)
    hc_wins = np.sum(gates <= 0.5)
    print(f"Thống kê nhanh:")
    print(f"   - Số lần ưu tiên PTM (WavLM/HuBERT): {ptm_wins} ({100*ptm_wins/len(gates):.2f}%)")
    print(f"   - Số lần ưu tiên Handcrafted (MFCC/Pitch): {hc_wins} ({100*hc_wins/len(gates):.2f}%)")
else:
    print("ℹChế độ hiện tại không sử dụng Gating. Bỏ qua bước phân tích này.")

## 6. Experiment Comparison

In [ ]:
import pandas as pd

# List all experiments
exp_base_dir = "./outputs/experiments"
if os.path.exists(exp_base_dir):
    experiments = []
    for exp_name_dir in sorted(os.listdir(exp_base_dir)):
        exp_path = os.path.join(exp_base_dir, exp_name_dir)
        results_file = os.path.join(exp_path, "results.json")
        if os.path.exists(results_file):
            with open(results_file, "r") as f:
                data = json.load(f)
                config = data.get("config", {})
                metrics = data.get("best_metrics", {})
                experiments.append({
                    "Experiment": exp_name_dir,
                    "Mode": config.get("mode", ""),
                    "Fusion": config.get("fusion_method", "N/A"),
                    "Feature": config.get("feature_mode", "N/A"),
                    "Best EER": f"{metrics.get('best_val_eer', 0):.2f}%",  # Đã đổi từ Loss sang EER
                    "MinDCF": f"{metrics.get('best_val_mindcf', 0):.4f}",
                    "Epochs": data.get("epochs_trained", 0),
                })
    
    if experiments:
        df = pd.DataFrame(experiments)
        print("\n" + "="*120)
        print("EXPERIMENT COMPARISON")
        print("="*120)
        print(df.to_string(index=False))
        print("="*120)
    else:
        print("No experiments found.")
else:
    print(f"Directory {exp_base_dir} does not exist yet.")